# Projeto Prático - Pré-processamento de Dados em Vendas

Este notebook implementa um pipeline de pré-processamento para dados de vendas, seguindo a linha trabalhada na disciplina: entendimento do negócio, ingestão, diagnóstico, limpeza, transformação e geração de uma base analítica confiável.

## Problema de negócio

Pergunta norteadora: como preparar uma base confiável para analisar o desempenho de vendas, o mix de produtos e o comportamento das movimentações ao longo do tempo?

A ideia é unir os arquivos brutos de movimentações e cadastro de produtos/serviços, tratar inconsistências e gerar uma base pronta para análises e futuras modelagens.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)


In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / 'dados').exists() and (PROJECT_ROOT.parent / 'dados').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / 'dados' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'dados' / 'processed'
FIGURES_DIR = PROJECT_ROOT / 'figures'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def carregar_movimentacoes(pasta: Path) -> pd.DataFrame:
    arquivos = sorted(pasta.glob('movimentacoes_*.json'))
    quadros = []
    for arquivo in arquivos:
        with arquivo.open(encoding='utf-8') as f:
            registros = json.load(f)
        quadro = pd.DataFrame(registros)
        quadro['_arquivo_origem'] = arquivo.name
        quadros.append(quadro)
    return pd.concat(quadros, ignore_index=True)

df_mov = carregar_movimentacoes(RAW_DIR)
df_prod = pd.read_csv(RAW_DIR / 'produtos_servicos_20260512.csv')

print('Movimentações:', df_mov.shape)
print('Produtos/serviços:', df_prod.shape)


## Diagnóstico inicial

Aqui a gente valida tipos, valores faltantes, duplicidades e alguns sinais básicos de qualidade.

In [ ]:
colunas_numericas_mov = [
    'valor_desconto_digitado',
    'valor_desconto_proporcional',
    'valor_frete_item',
    'qtd_item_movimentacao',
    'qtd_venda',
    'valor_unitario',
]

for coluna in colunas_numericas_mov:
    df_mov[coluna] = pd.to_numeric(df_mov[coluna], errors='coerce')

df_mov['data_emissao'] = pd.to_datetime(df_mov['data_emissao'], errors='coerce')
for coluna in ['status_item_cancelado']:
    df_mov[coluna] = df_mov[coluna].astype(str).str.lower().map({'true': True, 'false': False})

colunas_booleanas_prod = ['status_produto_servico', 'pesavel', 'vendavel', 'flag_item_ativo']
for coluna in colunas_booleanas_prod:
    df_prod[coluna] = df_prod[coluna].astype(str).str.lower().map({'true': True, 'false': False})

colunas_numericas_prod = [
    'percentual_cashback',
    'margem_lucro_aplicada_referencia',
    'limite_desconto_referencia',
    'percentual_comissao_referencia',
]
for coluna in colunas_numericas_prod:
    df_prod[coluna] = pd.to_numeric(df_prod[coluna], errors='coerce')

print('Nulos nas movimentações:')
display(df_mov.isna().sum().sort_values(ascending=False).head(10))
print('Nulos nos produtos/serviços:')
display(df_prod.isna().sum().sort_values(ascending=False).head(10))
print('Duplicadas nas movimentações:', df_mov.duplicated().sum())
print('Duplicadas nos produtos/serviços:', df_prod.duplicated(subset=['id_produto_servico']).sum())


In [ ]:
df_base = df_mov.merge(
    df_prod,
    on='id_produto_servico',
    how='left',
    suffixes=('_mov', '_prod')
)

texto_principal = [
    'descricao',
    'descricao_modelo',
    'descricao_situacao',
    'descricao_cfop',
    'tipo_item_descricao',
    'sub_grupo_referencia',
    'unidade_sigla',
]

for coluna in texto_principal:
    if coluna in df_base.columns:
        df_base[coluna] = df_base[coluna].fillna('desconhecido')

df_base = df_base.drop_duplicates().copy()
df_base = df_base.dropna(subset=['data_emissao', 'id_produto_servico', 'qtd_venda', 'valor_unitario']).copy()

df_base = df_base[df_base['tipo_transacao'].astype(str).str.upper().eq('VENDA')].copy()
df_base = df_base[df_base['direcao_estoque'].astype(str).str.upper().eq('SAIDA')].copy()
df_base['status_item_cancelado'] = df_base['status_item_cancelado'].fillna(False)
df_base = df_base[~df_base['status_item_cancelado']].copy()

df_base['codigo_barras'] = df_base['codigo_barras'].fillna('').astype(str).str.strip()
df_base['codigo_barras_tributavel'] = df_base['codigo_barras_tributavel'].fillna('').astype(str).str.strip()

df_base['receita_bruta_item'] = df_base['qtd_venda'] * df_base['valor_unitario']
df_base['desconto_total_item'] = df_base[['valor_desconto_digitado', 'valor_desconto_proporcional']].fillna(0).sum(axis=1)
df_base['receita_liquida_item'] = df_base['receita_bruta_item'] - df_base['desconto_total_item'] + df_base['valor_frete_item'].fillna(0)

q1 = df_base['receita_liquida_item'].quantile(0.25)
q3 = df_base['receita_liquida_item'].quantile(0.75)
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr
df_base['flag_outlier_receita'] = ~df_base['receita_liquida_item'].between(limite_inferior, limite_superior)

print('Base tratada:', df_base.shape)
print('Outliers de receita:', int(df_base['flag_outlier_receita'].sum()))
display(df_base[['data_emissao', 'descricao', 'qtd_venda', 'valor_unitario', 'receita_liquida_item']].head())


## Transformação de dados

Nesta etapa criamos atributos derivados, codificamos categorias e padronizamos variáveis numéricas para deixar a base pronta para análises futuras.

In [ ]:
dias_da_semana = {
    0: 'segunda',
    1: 'terca',
    2: 'quarta',
    3: 'quinta',
    4: 'sexta',
    5: 'sabado',
    6: 'domingo',
}

df_base['dia_semana'] = df_base['data_emissao'].dt.dayofweek.map(dias_da_semana)
df_base['mes_referencia'] = df_base['data_emissao'].dt.strftime('%Y-%m')
df_base['eh_final_de_semana'] = df_base['dia_semana'].isin(['sabado', 'domingo'])
df_base['faixa_receita'] = pd.cut(
    df_base['receita_liquida_item'],
    bins=[-np.inf, 0, 50, 100, np.inf],
    labels=['sem_receita', 'baixa', 'media', 'alta']
)

colunas_categoricas = ['tipo_item_descricao', 'unidade_sigla', 'dia_semana', 'faixa_receita']
base_modelagem = pd.get_dummies(df_base, columns=colunas_categoricas, drop_first=True)

colunas_norm = ['qtd_venda', 'valor_unitario', 'receita_bruta_item', 'receita_liquida_item']
scaler = StandardScaler()
base_modelagem[[f'{col}_z' for col in colunas_norm]] = scaler.fit_transform(base_modelagem[colunas_norm])

print('Base para modelagem:', base_modelagem.shape)
display(base_modelagem.head())


## Análises exploratórias finais

A ideia aqui é gerar indicadores simples para a narrativa do relatório final e do vídeo.

In [ ]:
top_produtos = (
    df_base.groupby('descricao', as_index=False)['receita_liquida_item']
    .sum()
    .sort_values('receita_liquida_item', ascending=False)
    .head(10)
)

receita_por_dia = (
    df_base.groupby('data_emissao', as_index=False)['receita_liquida_item']
    .sum()
    .sort_values('data_emissao')
)

print('Top 10 produtos por receita:')
display(top_produtos)

ax = top_produtos.sort_values('receita_liquida_item').plot(
    kind='barh',
    x='descricao',
    y='receita_liquida_item',
    legend=False,
    color='#2a9d8f'
)
ax.set_title('Top 10 produtos por receita líquida')
ax.set_xlabel('Receita líquida')
ax.set_ylabel('Produto')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'top_10_produtos_receita.png', dpi=150, bbox_inches='tight')
plt.show()

ax = receita_por_dia.plot(x='data_emissao', y='receita_liquida_item', marker='o', color='#264653')
ax.set_title('Receita líquida por dia')
ax.set_xlabel('Data')
ax.set_ylabel('Receita líquida')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'receita_por_dia.png', dpi=150, bbox_inches='tight')
plt.show()


## Conclusão

O pipeline consolida os dados brutos, corrige tipos, trata nulos, remove registros não aderentes ao caso de vendas, cria variáveis analíticas e entrega uma base pronta para análise e modelagem.

Próximos passos sugeridos:
- incluir `grupo.txt`;
- completar o texto do relatório final;
- gravar o vídeo de apresentação;
- registrar a base tratada em `dados/processed/`.

In [ ]:
saida_base = PROCESSED_DIR / 'base_vendas_tratada.csv'
saida_modelagem = PROCESSED_DIR / 'base_vendas_modelagem.csv'

df_base.to_csv(saida_base, index=False)
base_modelagem.to_csv(saida_modelagem, index=False)

print(f'Base tratada salva em: {saida_base}')
print(f'Base para modelagem salva em: {saida_modelagem}')
